In [1]:
!pip install fvcore av pytorchvideo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 49.3 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=36a1eb56769cecb2a7c903571775c37f93d3a3ba6a82c6819ee487da510d4c83
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=088b628743950e21e95e25b53f413919ff7719692a66e226c1978c63868e0a68
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel f

/kaggle/input/datasets/sundye/wlasl300/WLASL_300/

In [2]:
import os
import cv2
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATA_DIR = "/kaggle/input/datasets/sundye/wlasl300/WLASL_300/"
BATCH_SIZE = 8 
LR = 1e-4      
EPOCHS = 20
SAVE_PATH = "best_slowfast_wlasl300.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class WLASLDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.root_dir = os.path.join(root_dir, mode)
        self.mode = mode
        if not os.path.exists(self.root_dir):
            raise FileNotFoundError(f"Path not found: {self.root_dir}")
            
        self.classes = sorted(os.listdir(self.root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(self.root_dir, cls)
            if not os.path.isdir(cls_path):
                continue
            for vid in os.listdir(cls_path):
                if vid.endswith('.mp4'):
                    self.samples.append((os.path.join(cls_path, vid), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def _get_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        window_size = 64
        
        if self.mode == 'train':
            start_frame = np.random.randint(0, max(1, total_frames - window_size))
        else:
            start_frame = max(0, (total_frames - window_size) // 2)

        buffer = []
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        for _ in range(window_size):
            ret, frame = cap.read()
            if not ret:
                frame = buffer[-1] if len(buffer) > 0 else np.zeros((224, 224, 3), np.uint8)
            else:
                frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (224, 224))
            buffer.append(frame)
        cap.release()
        return np.array(buffer)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        raw_clip = self._get_frames(video_path)
        
        raw_clip = raw_clip.astype(np.float32) / 255.0
        raw_clip = (raw_clip - 0.45) / 0.225 

        slow_stride = 8
        slow_idx = np.arange(8) * slow_stride
        slow = torch.from_numpy(raw_clip[slow_idx]).permute(3, 0, 1, 2)
        
        fast_stride = 2 
        fast_idx = np.arange(32) * fast_stride
        fast = torch.from_numpy(raw_clip[fast_idx]).permute(3, 0, 1, 2)
        
        return [slow, fast], label

def build_slowfast_wlasl(num_classes=300):
    model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)
    in_features = model.blocks[6].proj.in_features
    model.blocks[6].proj = nn.Linear(in_features, num_classes)
    return model

train_dataset = WLASLDataset(DATA_DIR, 'train')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(WLASLDataset(DATA_DIR, 'val'), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(WLASLDataset(DATA_DIR, 'test'), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_slowfast_wlasl(num_classes=len(train_dataset.classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
best_val_acc = 0.0

classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

total_train_start = time.time()
epoch_durations = []

print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12} {'Time (s)':<10}")
print("-" * 85)

for epoch in range(EPOCHS):
    epoch_start_time = time.time()
    
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for inputs, labels in train_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = [i.to(device) for i in inputs]
            labels = labels.to(device)
            outputs = model(inputs)
            val_loss += criterion(outputs, labels).item()
            val_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
            val_total += labels.size(0)

    duration = time.time() - epoch_start_time
    epoch_durations.append(duration)
    avg_val_acc = 100 * val_correct / val_total
    
    save_msg = ""
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'class_to_idx': class_to_idx,
            'classes': classes,
            'epoch': epoch + 1,
            'best_val_acc': best_val_acc
        }
        torch.save(checkpoint, SAVE_PATH)
        save_msg = "*"

    print(f"{epoch+1:<8} {train_loss/len(train_loader):<12.4f} {100*train_correct/train_total:<12.2f} "
          f"{val_loss/len(val_loader):<12.4f} {avg_val_acc:<12.2f} {duration:<10.2f} {save_msg}")

total_train_end = time.time()
total_time_seconds = total_train_end - total_train_start
avg_time_per_epoch = sum(epoch_durations) / len(epoch_durations)

print("-" * 85)
print(f"Training Finished!")
print(f"Overall Training Time: {total_time_seconds / 60:.2f} minutes")
print(f"Average Time per Epoch: {avg_time_per_epoch:.2f} seconds")
print("-" * 85)

print("Running Final Evaluation on TEST set...")
checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_correct, test_total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = [i.to(device) for i in inputs]
        labels = labels.to(device)
        outputs = model(inputs)
        test_correct += (torch.max(outputs, 1)[1] == labels).sum().item()
        test_total += labels.size(0)

print(f"Final Test Accuracy: {100 * test_correct / test_total:.2f}%")

Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOWFAST_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/SLOWFAST_8x8_R50.pyth


100%|██████████| 264M/264M [00:01<00:00, 148MB/s]


Epoch    Train Loss   Train Acc    Val Loss     Val Acc      Time (s)  
-------------------------------------------------------------------------------------
1        5.6862       1.27         5.5558       1.33         882.02     *
2        5.1699       3.44         4.5363       6.33         886.70     *
3        4.0327       12.62        3.3717       20.75        887.28     *
4        2.9787       28.97        2.7692       31.19        887.29     *
5        2.1843       45.51        2.4051       41.29        886.40     *
6        1.4897       63.48        2.2358       46.50        888.23     *
7        0.9952       77.01        2.0868       50.17        887.63     *
8        0.6018       88.14        2.0823       49.94        888.37     
9        0.4077       92.90        1.9422       54.83        887.63     *
10       0.2907       94.87        2.0519       51.94        888.37     
11       0.1998       97.10        1.9382       51.72        888.10     
12       0.1570       97.63    